[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/deep-learning-diagnostics-and-improvement/blob/main/practice/16_conditioning_and_dit.ipynb)

# 16. Conditioning and DiT modulation

조건을 단순 concat하는 방식에서 FiLM, cross-attention, AdaLN/AdaLN-Zero로 확장한다.

**반복 형식:** 바닐라 PyTorch 실행 → profiler로 ATen/CUDA 연산 확인 → 필요할 때만 작은 텐서로 수학적 전개를 펼친다.


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("torch:", torch.__version__)


In [ ]:
from torch.profiler import profile, ProfilerActivity

def profile_call(name, fn, *args, **kwargs):
    activities = [ProfilerActivity.CPU]
    if torch.cuda.is_available():
        activities.append(ProfilerActivity.CUDA)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    with profile(
        activities=activities,
        record_shapes=True,
        profile_memory=True,
        with_stack=False,
    ) as prof:
        out = fn(*args, **kwargs)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    print(f"\n[{name}] top operators")
    sort_key = "self_cuda_time_total" if torch.cuda.is_available() else "self_cpu_time_total"
    print(prof.key_averages().table(sort_by=sort_key, row_limit=12))

    return out


## 1. Concatenation

feature와 condition을 feature 축에서 직접 붙인다.


In [ ]:
x = torch.tensor([[1., 2., 3.]], device=device)
c = torch.tensor([[10., 20.]], device=device)
print(torch.cat([x, c], dim=-1))


In [ ]:
_ = profile_call("concat conditioning", lambda: torch.cat([x,c],-1))


## 2. FiLM

condition에서 scale과 shift를 만들어 feature에 적용한다.


In [ ]:
feat = torch.tensor([[1., 2., 3., 4.]], device=device)
gamma = torch.tensor([[1., 0.5, 2., 1.]], device=device)
beta = torch.tensor([[0., 1., -1., 0.]], device=device)

film = gamma * feat + beta
print(film)


In [ ]:
_ = profile_call("FiLM", lambda: gamma*feat + beta)


## 3. Cross-attention

query와 key/value의 source가 다르다.


In [ ]:
q = torch.randn(1, 2, 3, 4, device=device)
k = torch.randn(1, 2, 5, 4, device=device)
v = torch.randn(1, 2, 5, 4, device=device)

out = F.scaled_dot_product_attention(q, k, v)
print(out.shape)


In [ ]:
_ = profile_call("cross attention", F.scaled_dot_product_attention, q, k, v)


## 4. AdaLN

condition으로 LayerNorm 이후 scale/shift를 만든다.


In [ ]:
x = torch.randn(1, 3, 8, device=device)
cond = torch.randn(1, 4, device=device)

norm = nn.LayerNorm(8, elementwise_affine=False).to(device)
mod = nn.Linear(4, 16).to(device)

scale, shift = mod(cond).chunk(2, dim=-1)
y = norm(x) * (1 + scale[:, None]) + shift[:, None]

print(y.shape)


In [ ]:
_ = profile_call("AdaLN", lambda: norm(x)*(1+scale[:,None])+shift[:,None])


## 5. AdaLN-Zero

residual branch의 output gate를 0에서 시작하는 구조를 최소 형태로 본다.


In [ ]:
gate = torch.zeros(1, 8, device=device)
branch = nn.Linear(8, 8, bias=False).to(device)
z = x + gate[:, None] * branch(y)

print("difference from input:", (z - x).abs().max().item())


In [ ]:
_ = profile_call("AdaLN-Zero residual", lambda: x + gate[:,None]*branch(y))


## References and provenance

**[16.1] FiLM**
- 출처: Perez et al., FiLM
- 이 노트북에서 가져온 부분: feature-wise scale/shift conditioning

**[16.2] Cross-attention**
- 출처: Transformer encoder-decoder / latent diffusion lineage
- 이 노트북에서 가져온 부분: condition source separated from query

**[16.3] DiT AdaLN-Zero**
- 출처: Peebles & Xie, Scalable Diffusion Models with Transformers
- 이 노트북에서 가져온 부분: adaptive LayerNorm modulation and zero-gated residual

**[16.4] MMDiT / modern DiT**
- 출처: Stable Diffusion 3 / FLUX / Krea / Anima families
- 이 노트북에서 가져온 부분: multimodal transformer conditioning variants
